# Procesamiento de datos mediante Apache Spark

## Autores

Eduardo Alarcón Navarro - 100472175@alumnos.uc3m.es

Ángela Ochoa Lavieja - 100562058@alumnos.uc3m.es

Lucía Velasco González - 100383335@alumnos.uc3m.es

Manuel Gómez-Plana Rodríguez - 100472310@alumnos.uc3m.es

# 1. Introducción

Este documento guarda la información relacionada con la elaboración de la práctica 1 de la asignatura Computación de Altas Prestaciones del Máster Universitario en Ciencia y Tecnología Informática. El notebook se divide en las siguientes secciones:

1. **Introducción**: se introduce la práctica.
2. **Problemas**: se introducen los posibles problemas planteados para realizar el estudio y el problema escogido para su análisis.
3. **Resolución**: se resuelve el problema mediante tres técnicas distintas: consultas SQL, concatenación de dataframes de Pandas y uso de RDDs.
4. **Análisis**: se analizan los resultados obtenidos de los tres problemas para determinar aspectos como la solución más rápida, el speedup obtenido mediante cada técnica o la escalabilidad de las soluciones.
5. **Conclusiones**: se concluye el trabajo con unos pensamientos finales.

# Problemas



## Problemas Planteados

## Problema escogido

# Resolución

In [14]:
# Importado de Spark
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql.functions import *

# Creación de la sesión de spark
spark = SparkSession.builder.master('local[*]').getOrCreate()
#spark.conf.set('spark.cores.max', '1')
#spark.conf.set('spark.driver.memory','1000M')

sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=pyspark-shell>

In [2]:
zone_map = spark.read.format("csv").option("inferSchema", "true").option("header", "true").option("mode", "DROPMALFORMED").load("content/taxi_zone_lookup.csv")
zone_map.show()
zone_map: dict[int, list[str]] = zone_map.toPandas().set_index('LocationID').T.to_dict('list')

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [15]:
df = spark.read.format("parquet").option("inferSchema", "true").option("timestampFormat","yyyy-MM-dd HH:mm:ss").option("header", "true").option("mode", "DROPMALFORMED").load("content/yellow_tripdata_2025-01.parquet")
df.printSchema()
df.show(5)
type(df)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

pyspark.sql.classic.dataframe.DataFrame

Limpieza de datos

In [4]:
# df = df.na.drop()

In [16]:
df = df.filter(df.trip_distance != 0)
df = df.filter(df.tpep_pickup_datetime != df.tpep_dropoff_datetime)

In [6]:
TABLE_NAME = 'taxi_routes'
df.createOrReplaceTempView(TABLE_NAME)
SQL = f"""SELECT HOUR(tpep_pickup_datetime) AS hour,
            ROUND(AVG(trip_distance / DATE_DIFF(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime))*1.6*3600, 2) as vel_media_km_h 
            FROM {TABLE_NAME} 
            GROUP BY hour 
            SORT BY hour """
sql_result = spark.sql(SQL)

In [7]:
sql_result.show(24)

+----+--------------+
|hour|vel_media_km_h|
+----+--------------+
|   0|          39.0|
|   1|         39.91|
|   2|         23.37|
|   3|         24.48|
|   4|         45.45|
|   5|         658.9|
|   6|        180.16|
|   7|         43.73|
|   8|         68.31|
|   9|         45.11|
|  10|         29.39|
|  11|         16.25|
|  12|         17.44|
|  13|         27.22|
|  14|         47.12|
|  15|         28.21|
|  16|         32.59|
|  17|         36.57|
|  18|         31.95|
|  19|          28.1|
|  20|         30.75|
|  21|         64.85|
|  22|         33.28|
|  23|         40.29|
+----+--------------+



## Usando dataframes

In [43]:
df2=df.withColumn('DiffInSeconds', (col('tpep_dropoff_datetime') - col('tpep_pickup_datetime')).cast('long'))
df3 = df2.withColumn('vel_km_h', try_divide('trip_distance','DiffInSeconds')*5760)
df4 = df3.withColumn('start_hour', hour(col('tpep_pickup_datetime')))
df4.select('start_hour', 'vel_km_h').groupBy("start_hour").avg('vel_km_h').show(truncate=False)

AttributeError: 'GroupedData' object has no attribute 'select'

# Análisis

In [38]:
df4.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------+------------------+----+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|DiffInSeconds|          vel_km_h|hour|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------+------------------+----+
|       1| 2025-

# Conclusiones